# 09 · The A2A Protocol

*How an agent gets another **agent** to do the work.*

Notebook 08 ended with MCP: a tool can live in another process, and the model
cannot tell the difference. That is a big step, but the thing on the other end
of the wire was still just a **function**. You call it with typed arguments,
it returns a value, it has no opinions.

**A2A (Agent-to-Agent) changes what is on the other end.** The callee is
itself an agent — its own LLM, its own tools, its own reasoning loop. So the
call changes shape too:

| | MCP | A2A |
|---|---|---|
| Other end is | a function | an agent |
| You send | typed arguments matching a JSON Schema | a **brief** in natural language |
| You get back | a return value | a **Task** with a prose reply *and* structured artifacts |
| Duration | milliseconds | seconds to minutes · it is thinking |
| Contract published as | JSON Schema | an **Agent Card** |

That last row is the one to hold on to. An MCP tool tells you *exactly* what
arguments it accepts. An A2A agent tells you, in prose and with examples,
*what it is good at* — because you cannot write a JSON Schema for
"plan me a good trip."

In this notebook we talk to the **live Session 6 mesh** from outside. Nothing
here is a simulation: real cards, real JSON-RPC, real agents thinking.

1. Discovery — fetch an Agent Card and read the contract.
2. The whole mesh's capability surface, in one loop.
3. Invocation — build the JSON-RPC envelope by hand and send it.
4. Two errors worth meeting on purpose.
5. The Task lifecycle, and why results come back as Artifacts not Messages.
6. What the SDK wrapper hides, and where it breaks.

> **Prerequisite.** The Session 6 stack must be running:
> `docker compose up -d` in the repo root. Cell 1 checks.

In [ ]:
import os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import json
import httpx

# Host-mapped ports. Inside the Docker network these agents address each
# other as http://flight-agent:8010/ etc. From here — outside the network —
# we reach the same containers on localhost. That distinction bites us in
# Step 8, and it is worth remembering now.
AGENTS = {
    "flight-agent":    "http://localhost:8010",
    "hotel-agent":     "http://localhost:8011",
    "itinerary-agent": "http://localhost:8012",
    "critic-agent":    "http://localhost:8015",
    "todo-agent":      "http://localhost:8016",
    "research-agent":  "http://localhost:8018",
}

def is_up(base: str) -> bool:
    try:
        return httpx.get(f"{base}/health", timeout=3.0).status_code == 200
    except Exception:
        return False

status = {name: is_up(base) for name, base in AGENTS.items()}
for name, ok in status.items():
    print(f"  {'up  ' if ok else 'DOWN'}  {name:16} {AGENTS[name]}")

UP = all(status.values())
if not UP:
    print("\n  Some agents are unreachable. Start the mesh with:")
    print("    docker compose up -d")

## Step 1 — Discovery: the Agent Card

Every A2A agent publishes a card at a well-known path:

```
GET /.well-known/agent-card.json
```

That is the entire discovery mechanism. No registry, no service catalogue,
no broker. If you can reach the agent, you can ask what it does — the same
way `robots.txt` or an OpenID configuration document works.

In [ ]:
card = httpx.get(f"{AGENTS['flight-agent']}/.well-known/agent-card.json",
                 timeout=10.0).json()

print("name       :", card["name"])
print("version    :", card["version"])
print("streaming  :", card["capabilities"].get("streaming"))
print("input/output:", card["defaultInputModes"], "->", card["defaultOutputModes"])
print()
print("skills:")
for s in card["skills"]:
    print(f"  · {s['id']:16} {s['name']}")
    print(f"      tags: {', '.join(s['tags'])}")
    for ex in s["examples"]:
        print(f"      e.g. {ex}")

## Step 2 — A card is a contract written in prose

Look at what a skill actually carries: an `id`, a `name`, a `description`,
some `tags`, and — the important part — **`examples`**.

There is no argument schema anywhere. Compare with the MCP tool from
notebook 08, where `create_event` published exact typed parameters
(`time_min: string`, `max_results: integer`).

That difference is not sloppiness, it is the point:

- An MCP tool is **deterministic**. Same arguments, same behaviour. A schema
  fully describes it.
- An A2A agent is **judgement-shaped**. You are asking a specialist to
  handle something. The useful contract is *what it is good at* and *what a
  well-formed request looks like* — which is exactly what examples convey.

So the `examples` list is not documentation for humans. In this codebase it
is fed straight into the prompt of the calling agent. Notebook 10 shows that
machinery; here just notice that the card is written to be **read by a model**.

In [ ]:
# The full description block on the card — this is prose aimed at the caller.
print(card["description"])

## Step 3 — The whole mesh, in one loop

Six specialists, each publishing its own card. This loop is the complete
capability surface of the mesh — and note that nothing here consults a
central registry. We iterate over URLs we know and ask each one directly.

In [ ]:
def fetch_card(base: str) -> dict:
    return httpx.get(f"{base}/.well-known/agent-card.json", timeout=10.0).json()

cards = {name: fetch_card(base) for name, base in AGENTS.items() if status[name]}

total_skills = 0
for name, c in cards.items():
    skills = c.get("skills") or []
    total_skills += len(skills)
    print(f"{name:16} {len(skills)} skill(s): {', '.join(s['id'] for s in skills)}")

print(f"\n{len(cards)} agents · {total_skills} skills advertised across the mesh")

## Step 4 — The address on the card is not your address

Here is a subtlety that costs people an afternoon.

The card advertises where to reach the agent, in `supportedInterfaces`. But
the agent publishes the address that its **peers** use — the one that works
inside the network it lives in. Our agents run in Docker Compose, so they
advertise Docker DNS names.

We are outside that network. The card is telling the truth; it is just not
telling it to us.

In [ ]:
iface = card["supportedInterfaces"][0]
print("card advertises :", iface["url"])
print("binding         :", iface["protocolBinding"], "· protocol", iface["protocolVersion"])
print("we actually use :", AGENTS["flight-agent"] + "/")
print()
print("Same container. Two names. Only one of them resolves from here.")

## Step 5 — Invocation: the JSON-RPC envelope, by hand

A2A rides on **JSON-RPC 2.0**. One POST to the agent's root URL, with a
`method` and `params`. We will build it by hand rather than reach for the SDK,
because the envelope is small enough to read and you should see it once.

Two details that are easy to get wrong, and both fail loudly (Step 6):

- The method is **`SendMessage`** — proto-style naming, A2A v1.0.
- You must send an **`A2A-Version: 1.0`** header. Omit it and the server
  assumes `0.3` and rejects you.

The `role` is `ROLE_USER` even though the caller is another agent. From the
callee's point of view, whoever is asking occupies the user role.

In [ ]:
import time

A2A_HEADERS = {"Content-Type": "application/json", "A2A-Version": "1.0"}

def a2a_send(base: str, brief: str, *, request_id: str = "nb-09") -> dict:
    """POST one A2A SendMessage and return the raw JSON-RPC response."""
    envelope = {
        "jsonrpc": "2.0",
        "id": request_id,
        "method": "SendMessage",
        "params": {
            "message": {
                "messageId": f"msg-{request_id}",
                "role": "ROLE_USER",
                "parts": [{"text": brief}],
            }
        },
    }
    # read timeout is generous · the agent runs an LLM plus MCP calls
    r = httpx.post(base + "/", headers=A2A_HEADERS, json=envelope,
                   timeout=httpx.Timeout(connect=5.0, read=180.0,
                                         write=30.0, pool=10.0))
    r.raise_for_status()
    return r.json()

BRIEF = ("Search BLR to NRT on 2026-10-15 for 2 adults, "
         "prefer non-stop, under 150000 INR per leg")

t0 = time.time()
resp = a2a_send(AGENTS["flight-agent"], BRIEF)
elapsed = time.time() - t0

print(f"round trip: {elapsed:.1f}s")
print("top-level keys:", list(resp.keys()))
print("result keys   :", list(resp["result"].keys()))

### That took seconds, not milliseconds

Sit with that number. An MCP tool call is a function invocation — it returns
in milliseconds. This one took long enough that you noticed, because on the
other side an entire agent woke up, called its LLM, made its own MCP calls to
`mcp-airline`, ranked twelve options, and wrote a recommendation.

Every timeout in `shared/a2a_helpers/client.py` exists because of this. The
default httpx read timeout is 5 seconds; the helper raises it to 180.

## Step 6 — Two errors worth meeting on purpose

Both of these are protocol-level rejections, and both have a very specific
shape. Meeting them here means recognising them instantly later.

In [ ]:
def show_error(label: str, *, method: str, headers: dict):
    body = {
        "jsonrpc": "2.0", "id": "err-probe", "method": method,
        "params": {"message": {"messageId": "m-err", "role": "ROLE_USER",
                               "parts": [{"text": "ping"}]}},
    }
    r = httpx.post(AGENTS["flight-agent"] + "/", headers=headers,
                   json=body, timeout=15.0)
    err = r.json().get("error", {})
    print(f"{label}")
    print(f"  code    : {err.get('code')}")
    print(f"  message : {err.get('message')}")
    print()

# 1 · the v0.3 spelling of the method name
show_error("Using the old 'message/send' method name:",
           method="message/send", headers=A2A_HEADERS)

# 2 · no version header · server assumes 0.3
show_error("Omitting the A2A-Version header:",
           method="SendMessage",
           headers={"Content-Type": "application/json"})

`-32601 Method not found` is plain JSON-RPC: the verb does not exist.
The A2A spec renamed these between 0.3 and 1.0, so half the blog posts you
will find use `message/send`.

`-32009 VERSION_NOT_SUPPORTED` is A2A-specific and more interesting: an
**absent** header is not treated as "no opinion", it is treated as
`0.3`. The protocol chose a default rather than a negotiation, so an old
client talking to a new server fails cleanly instead of half-working.

## Step 7 — The Task lifecycle

A2A does not return a value. It returns a **Task** — an object with an id,
a lifecycle state, and accumulated outputs. Our agents run to completion in
one call, so we see the final state directly; a long-running agent would let
you poll `GetTask` or subscribe for updates.

In [ ]:
task = resp["result"]["task"]

print("task id   :", task["id"])
print("context id:", task["contextId"])
print("state     :", task["status"]["state"])
print("timestamp :", task["status"]["timestamp"])
print()
print("artifacts :", [a["name"] for a in task.get("artifacts", [])])
print(resp["result"])

## Step 8 — Messages carry prose · Artifacts carry data

This split is mandated by the spec (§3.7: *"Messages SHOULD NOT be used to
deliver task outputs. Results SHOULD BE returned using Artifacts"*), and it
is the design decision worth stealing even if you never write A2A.

- **`status.message`** — what the agent would *say*. Written for a reader.
- **`artifacts`** — what the agent *found*. Written for a program.

Without the split you get the failure mode everyone building agents hits: the
supervisor LLM reads a wall of structured data, tries to summarise it, and
hallucinates a price. Here the supervisor paraphrases the prose, and the UI
renders cards straight from the artifact. Neither has to parse the other.

In [ ]:
prose = "".join(p.get("text", "") for p in task["status"]["message"]["parts"])
print("── what the agent SAYS ──")
print(prose)

In [ ]:
art = task["artifacts"][0]
rows = art["parts"][0]["data"]["result"]

print(f"── what the agent FOUND ── artifact '{art['name']}' · {len(rows)} rows\n")
print(f"{'flight':9} {'airline':18} {'stops':>5} {'hrs':>5} {'total INR':>11}")
for r in rows[:6]:
    print(f"{r['flight_id']:9} {r['airline'][:18]:18} {r['stops']:>5.0f} "
          f"{r['duration_hours']:>5.1f} {r['price_total_inr']:>11,.0f}")
print(f"... {len(rows) - 6} more")

### The float that surprises everyone

Print `rows[0]["stops"]` and you get `1.0`, not `1`. Nobody wrote a float.

Artifact data travels as `google.protobuf.Value`, and that type has exactly
one number kind: **double**. So every integer round-trips as a float —
`pax`, `stops`, `price_total_inr`, all of them.

This is invisible in JavaScript, where `Number` is always a double, which is
why the frontend never noticed. A Python consumer has to cast explicitly.
It is documented in `shared/a2a_helpers/client.py`, and it is the kind of
thing you only learn by printing.

In [ ]:
r = rows[0]
for k in ("flight_id", "stops", "pax", "price_total_inr", "duration_hours"):
    print(f"  {k:18} {r[k]!r:>14}   ({type(r[k]).__name__})")

## Step 9 — What the SDK does, and where it breaks

Doing this by hand was for teaching. In the real planner, all of the above
collapses to one line:

```python
result = await delegate_to_a2a_agent(agent_url, brief)
# -> {"text": "...", "artifacts": [{"name": ..., "data": ...}, ...]}
```

`shared/a2a_helpers/client.py` wraps the a2a-sdk `Client`, drives the task to
completion, and reduces everything to that `{text, artifacts}` envelope — the
supervisor never sees a JSON-RPC envelope at all.

Now watch it fail from here, for the reason we set up in Step 4. The SDK
does discovery *properly*: it fetches the card and then honours the URL the
card advertises. That URL is `http://flight-agent:8010/`, which does not
resolve outside Docker. Our hand-rolled version worked precisely **because**
it ignored the card and used the address we already had.

In [ ]:
import asyncio
from shared.a2a_helpers import delegate_to_a2a_agent

try:
    out = await delegate_to_a2a_agent(AGENTS["flight-agent"] + "/", "Hold the ANA NH814 for 2 adults")
    print("text:", out["text"][:160])
    print("artifacts:", [a["name"] for a in out["artifacts"]])
except Exception as exc:
    print(f"{type(exc).__name__}: {str(exc)[:200]}")
    print()
    print("Expected, from outside the Docker network. The SDK followed the card's")
    print("advertised URL. Inside the mesh — where the planner runs — this is the")
    print("call that works, and the hand-rolled one would be the fragile choice.")

## Recap

1. **A2A puts an agent on the far end of the call**, not a function. So the
   request is a natural-language *brief* and the reply is a *Task*, not a
   return value.
2. **Discovery is a well-known URL.** `GET /.well-known/agent-card.json`.
   No registry, no broker.
3. **The card is prose plus examples, not a schema** — because you cannot
   schema-define "recommend a good flight." Those examples get fed to the
   calling model; notebook 10 shows how.
4. **The card advertises the address peers use.** Inside Docker that is a
   service name. The SDK honours it, which is right in production and
   wrong from your laptop.
5. **`SendMessage` + `A2A-Version: 1.0`.** A missing version header means
   `0.3`, not "unspecified".
6. **Messages are prose, Artifacts are data** (spec §3.7). This is what keeps
   the supervisor from hallucinating prices out of a table.
7. **Artifact numbers are protobuf doubles.** `1` comes back as `1.0`.

MCP gave the agent *capabilities*. A2A lets agents **delegate to each other**.
Notebook 10 puts both in one process: a supervisor with 12 tools — 6 MCP,
6 A2A — that never had any of the six sub-agents' capabilities written down.

---

### Exercises

1. Send the *hold* brief from Step 5's card examples instead of the search
   brief. Which artifact name comes back, and how does the prose change?
2. Call `delegate_to_critic_agent`'s agent directly (`:8015`) with an
   itinerary to review. Compare its artifacts to the flight agent's.
3. Send a deliberately vague brief — `"book me something nice"`. Does the
   agent ask a question, guess, or fail? That behaviour is the sub-agent's
   own prompt, not the protocol.
4. Add `"blocking": False` to `params` and see whether this agent supports
   streaming. Its card says `streaming: false` — does the server agree?
5. Reach the same agent through the REST binding instead of JSON-RPC
   (`create_rest_routes` mounts it). Same task, different envelope.